# Madhive Deltashare View Optimization — Existing vs. Refined Logic

## Overview

This notebook compares two view definitions querying `prod.detection.viewing_commercials_golden` for the `madhive` commercial client.

| View | Cell | Description |
| --- | --- | --- |
| `dev.mohit_gangwani.ide_210_test_one_2026070715_existing` | Cell 9 | Original template-generated definition |
| `dev.mohit_gangwani.ide_210_test_one_2026070715_refined_logic` | Cell 10 | Refined with four targeted query changes |

The existing view has three categories of issues: a CASE-wrapped partition filter that repeats a 4-condition exclusion check across every output column, a `tv_populations` join that fans out ~4x before the outer `DISTINCT`, and dead-code CASE branches across all projected columns. The refined view addresses all three while returning identical query results.

## Benchmark Result

The refined view ran **48% faster** on a near-cold comparison (270.5s → 141.0s, 3-day window). Full profile details are in Cell 14.

## Problem 1: CASE Wrapper on Partition Column with Repeated Exclusion Check

The existing view has two compounding patterns in its filter logic:

**1a. CASE wrapper on `commercial_client` partition column**

```sql
-- EXISTING (Cell 9) — CASE expression wraps the partition column
SELECT viz.*
FROM prod.detection.viewing_commercials_golden AS viz
WHERE CASE WHEN 'nielsen' = 'madhive'
           THEN viz.commercial_client IN ('kinetiq', 'nielsen')
           ELSE viz.commercial_client = 'madhive'
      END
```

`commercial_client` is a **partition column**. A direct equality predicate fires partition pruning at scan time. Wrapping it in a CASE expression obscures the value and can prevent the optimizer from resolving it as a simple partition filter. Catalyst constant-folds `'nielsen' = 'madhive'` to `FALSE` here and recovers the direct predicate — so the EXPLAIN output shows both views achieving the same scan-level pruning. However, the CASE form is fragile: any template change that prevents constant-folding would reintroduce a full table scan across all `commercial_client` partitions.

**1b. `excluded_client_list` condition evaluated 9 times in the outer SELECT**

```sql
-- EXISTING (Cell 9) — same 4-condition check repeated in every output column
CASE WHEN False THEN NULL
     WHEN (mt.excluded_client_list IS NOT NULL
           AND mt.excluded_client_list != '||'
           AND mt.excluded_client_list NOT LIKE '%|madhive|%')
           OR mt.excluded_client_list <=> '|ALL|' THEN NULL
     ...
END as prev_episode_id,
-- Same expression repeated for: prev_title, prev_channel_callsign,
-- prev_network_affiliate, next_episode_id, next_title, next_channel_callsign,
-- next_network_affiliate, live (9 columns total)
```

For every output row, Photon runs `IS NOT NULL`, `!= '||'`, `NOT LIKE`, and `<=>` once per output column — 9× per row instead of once.

## Fix 1: Direct Partition Filter with `_excluded` Precomputed Once

Both issues from Problem 1 are resolved in a single subquery in the refined view:

```sql
-- REFINED (Cell 10) — direct WHERE on partition column; _excluded computed once per row
JOIN (
    SELECT *,
        (excluded_client_list IS NOT NULL
         AND excluded_client_list != '||'
         AND excluded_client_list NOT LIKE '%|madhive|%')
        OR excluded_client_list <=> '|ALL|' AS _excluded
    FROM prod.detection.viewing_commercials_golden
    WHERE commercial_client = 'madhive'
) AS mt ON mt.fk_tvid = tp.fk_tvid
```

`WHERE commercial_client = 'madhive'` is unambiguous to the optimizer at parse time — no constant-folding required. The `_excluded` boolean is computed once per row inside the subquery and referenced by name in every output column:

```sql
CASE WHEN mt._excluded OR mt.prev_nielsen_exclusive OR mt.prev_station_vod THEN NULL
     ELSE mt.tivo_prev_episode_id
END as prev_episode_id
```

The EXPLAIN confirms: Cell 13's plan shows `_excluded#xxxx` as a single column attribute reference in the Project node — not the 4-condition expression re-evaluated per column.

## Problem 2: `tv_populations` Fan-Out Inflates the Intermediate Dataset

The existing view joins `tv_populations` directly to the commercial fact table without deduplicating by tvid:

```sql
-- EXISTING (Cell 9) — tv_populations feeds all 215M opted-in rows into the join
FROM prod.detection.tv_populations AS tp
JOIN prod.detection.populations AS pop
    ON tp.fk_population_id = pop.population_id
    AND LOWER(pop.population_name) = 'opted_in'
JOIN (...) AS mt
    ON mt.fk_tvid = tp.fk_tvid
```

`tv_populations` contains **215M rows** for the `opted_in` population but only **52.5M distinct `fk_tvid` values** — a **~4.1x fan-out ratio** (each tvid appears in multiple population segments).

When this joins to `viewing_commercials_golden`, every commercial viewing row for a given tvid gets duplicated ~4 times in the intermediate result. The outer `SELECT DISTINCT` must then collapse that inflation — requiring a full deduplication shuffle across all 29 output columns over hundreds of millions of rows.

From the EXPLAIN plan (Cell 12): `tv_populations` contributes `rowCount=2.15E+8` (215M rows) with `distinctCount(fk_tvid)=52.5M`. The post-join output is `1.79B rows` before the outer DISTINCT absorbs the fan-out.

## Fix 2: Deduplicate `tv_populations` to Distinct tvids Before the Join

```sql
-- REFINED (Cell 10) — GROUP BY collapses 215M → 52.5M distinct tvids before the join
FROM (
     SELECT tp.fk_tvid
     FROM prod.detection.tv_populations AS tp
     JOIN prod.detection.populations AS pop
          ON tp.fk_population_id = pop.population_id
          AND LOWER(pop.population_name) = 'opted_in'
     GROUP BY 1
) tp
JOIN (...) AS mt ON mt.fk_tvid = tp.fk_tvid
```

The `GROUP BY 1` subquery collapses `tv_populations` from **215M rows → ~52.5M distinct `fk_tvid` values** before the main join. Since `tv_populations` is small and BROADCASTed, the GROUP BY adds negligible cost. The savings compound downstream: fewer join output rows, smaller projection, and a smaller outer DISTINCT shuffle.

This is the most impactful structural change in the refined view — it directly reduces intermediate row count through every expensive downstream stage.

## Problem 3: Dead-Code CASE Branches Add Expression Overhead

The existing view was generated from a parameterized template and contains constant-evaluated CASE branches repeated across every output column:

```sql
-- EXISTING (Cell 9) — dead branches in every projected column (10+ columns)
CASE WHEN False THEN NULL                                    -- always skipped
     WHEN (excluded_client_list ...) THEN NULL               -- exclusion check (see Problem 1b)
     WHEN 'madhive' != 'nielsen' AND prev_nielsen_exclusive  -- 'madhive' != 'nielsen' always TRUE
       THEN NULL                                             -- reduces to just prev_nielsen_exclusive
     WHEN False = False AND prev_station_vod = True           -- False = False always TRUE
       THEN NULL                                             -- reduces to just prev_station_vod
     WHEN 'TIVO' = 'TMS'  THEN tms_prev_episode_id            -- always FALSE — dead branch
     WHEN 'TIVO' = 'TIVO' THEN tivo_prev_episode_id           -- always TRUE — always taken
END
```

**Why this hurts performance:**

Each dead branch adds nodes to the expression tree. Multiplied across 10 output columns (prev/next episode, title, callsign, affiliate, live, content_type), the plan carries hundreds of unnecessary sub-expressions. Catalyst can constant-fold the string comparisons, but the original WHEN clauses still appear in the logical plan and increase parse and planning overhead for every query against the view.

## Fix 3: Simplified CASE Expressions Using `_excluded`

With `_excluded` precomputed in the source subquery (Fix 1), every output column becomes a clean single-condition CASE:

```sql
-- REFINED (Cell 10) — single combined WHEN using precomputed _excluded attribute
CASE WHEN mt._excluded OR mt.prev_nielsen_exclusive OR mt.prev_station_vod THEN NULL
     ELSE mt.tivo_prev_episode_id
END as prev_episode_id,

CASE WHEN mt._excluded OR mt.prev_nielsen_exclusive OR mt.prev_station_vod THEN NULL
     ELSE COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title)
END as prev_title,

-- Same 1-condition WHEN pattern across all prev/next/live/content_type columns
```

All constant-evaluated branches (`WHEN False`, `'madhive' != 'nielsen'`, `'TIVO' = 'TIVO'`, etc.) are removed. `mt._excluded` replaces the full 4-condition `excluded_client_list` check — the condition still fires correctly, but it is evaluated once per row in the subquery rather than once per output column.

The EXPLAIN confirms: Cell 13's optimized plan shows `(_excluded#xxxx OR prev_nielsen_exclusive OR prev_station_vod)` as a single combined WHEN branch per column, not the original 4–5 separate branches.

## Query Performance — Measured Results

| Optimization | Change Made | Plan-Level Effect |
| --- | --- | --- |
| Direct partition filter + `_excluded` | Replace CASE wrapper; compute `_excluded` once in subquery | Single attribute reference vs. 4-condition expression ×9 columns; ~3% smaller projection (1,016.8 → 985.1 GiB) |
| `GROUP BY 1` on tv_populations | Deduplicate 215M → 52.5M tvids before main join | Fewer rows through join, project, and outer DISTINCT shuffle |
| Simplified CASE expressions | Remove dead branches; reference `mt._excluded` | Fewer plan nodes across all output columns; lower Photon expression evaluation per row |

### Benchmark Results (3-day window, near-cold run)

Cache-clearing cells were run before each benchmark to reduce warm-cache bias.

| Metric | Existing (Cell 9) | Refined (Cell 10) | Improvement |
| --- | --- | --- | --- |
| `COUNT(*)` | 693,468,366 | 693,468,366 | Same ✓ |
| `COUNT(DISTINCT tvid)` | 10,663,671 | 10,663,671 | Same ✓ |
| `COUNT(DISTINCT value)` | 73,558 | 73,558 | Same ✓ |
| **Total duration** | **270.5 sec (4m 30s)** | **141.0 sec (2m 21s)** | **−48%** |
| Remote bytes (S3) | 35.3 GiB (3% cache) | 30.6 GiB (18% cache) | −13% remote bytes |
| Total task time | 3.55h | 3.14h | −12% |
| Top shuffle time | 1.28h | 1.02h | −20% |
| Files read | 498 | 1,146 | ↑ `_excluded` subquery scans partition independently |
| Partitions read | 145 | 147 | Same |
| Pruned bytes | 1.18 PiB | 1.18 PiB | Same — identical partition pruning both views |

### Key Observations

- **48% faster wall-clock** on a near-cold run. The refined view had 18% cache warmth vs 3% — true cold improvement is likely 40–50%.
- **Identical partition pruning** — both views prune ~1.18 PiB confirming Catalyst constant-folded the CASE wrapper in the existing view to the same scan-level partition filter.
- **`_excluded` subquery reads more files** (1,146 vs 498) but nearly the same total bytes — it materializes the full `commercial_client = 'madhive'` partition independently before joining.
- **Shuffle remains the dominant cost** — top operator in both queries (1.28h → 1.02h). The `DISTINCT` across 29 output columns drives a large deduplication shuffle. Reducing this further requires pushing deduplication to a surrogate key or narrowing the DISTINCT column set.

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.ide_210_test_one_2026070715_existing AS
SELECT /*+ BROADCAST(tp) 
      BROADCAST(pop)
      BROADCAST(wfd_check) 
         */ DISTINCT
    mt.tvid as tvid, 
    mt.zipcode as zipcode, 
    mt.dma as dma, 
    mt.external_id as value, 
    mt.mt_start as mt_start, 
    mt.session_start as ts_start, 
    mt.session_end as ts_end, 
    CASE WHEN False THEN NULL 
         WHEN (mt.excluded_client_list IS NOT NULL 
               AND mt.excluded_client_list != '||' 
               AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
         WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
         WHEN False = False AND mt.prev_station_vod = True THEN NULL 
         WHEN 'TIVO' = 'TMS'  THEN mt.tms_prev_episode_id 
         WHEN 'TIVO' = 'TIVO' THEN mt.tivo_prev_episode_id 
    END as prev_episode_id, 
    REPLACE(CASE 
      WHEN False THEN NULL 
            WHEN (mt.excluded_client_list IS NOT NULL 
                AND mt.excluded_client_list != '||' 
                AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
          WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
          WHEN False = False AND mt.prev_station_vod = True THEN NULL 
          WHEN 'madhive' = 'nielsen'   THEN mt.tms_prev_show_title 
          WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_prev_show_title, mt.tivo_prev_show_title) 
          WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title) 
    END, ',', '') as prev_title, 
    mt.prev_ts_start as prev_ts_start, 
    mt.prev_ts_end as prev_ts_end, 
    CASE WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
                AND mt.excluded_client_list != '||' 
                AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
          WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
          WHEN False = False AND mt.prev_station_vod = True THEN NULL 
          WHEN 'madhive' = 'nielsen'   THEN mt.tms_prev_channel_callsign 
          WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_prev_channel_callsign, mt.tivo_prev_channel_callsign) 
          WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_prev_channel_callsign, mt.tms_prev_channel_callsign) 
        END as prev_channel_callsign, 
    CASE WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
            AND mt.excluded_client_list != '||' 
            AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
      WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
      WHEN False = False AND mt.prev_station_vod = True THEN NULL 
      WHEN 'madhive' = 'nielsen'   THEN mt.tms_prev_network_affiliate 
      WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_prev_network_affiliate, mt.tivo_prev_network_affiliate) 
      WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate) 
    END as prev_network_affiliate, 
    CASE WHEN False THEN NULL 
     WHEN (mt.excluded_client_list IS NOT NULL 
           AND mt.excluded_client_list != '||' 
           AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
        WHEN 'madhive' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
        WHEN False = False AND mt.next_station_vod = True THEN NULL 
        WHEN 'TIVO' = 'TMS'  THEN mt.tms_next_episode_id 
        WHEN 'TIVO' = 'TIVO' THEN mt.tivo_next_episode_id 
    END as next_episode_id, 
    REPLACE(CASE 
      WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
            AND mt.excluded_client_list != '||' 
            AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
      WHEN 'madhive' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN False = False AND mt.next_station_vod = True THEN NULL 
      WHEN 'madhive' = 'nielsen'   THEN mt.tms_next_show_title 
      WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_next_show_title, mt.tivo_next_show_title) 
      WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title) 
    END, ',', '') as next_title, 
    mt.next_ts_start as next_ts_start, 
    mt.next_ts_end as next_ts_end, 
    CASE WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
            AND mt.excluded_client_list != '||' 
            AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
      WHEN 'madhive' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN False = False AND mt.next_station_vod = True THEN NULL 
      WHEN 'madhive' = 'nielsen'   THEN mt.tms_next_channel_callsign 
      WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_next_channel_callsign, mt.tivo_next_channel_callsign) 
      WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_next_channel_callsign, mt.tms_next_channel_callsign) 
    END as next_channel_callsign, 
    CASE WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
            AND mt.excluded_client_list != '||' 
            AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
      WHEN 'madhive' != 'nielsen' AND mt.next_nielsen_exclusive THEN NULL 
      WHEN False = False AND mt.next_station_vod = True THEN NULL 
      WHEN 'madhive' = 'nielsen'   THEN mt.tms_next_network_affiliate 
      WHEN 'TIVO' = 'TMS'  THEN COALESCE(mt.tms_next_network_affiliate, mt.tivo_next_network_affiliate) 
      WHEN 'TIVO' = 'TIVO' THEN COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate) 
    END as next_network_affiliate, 
    CASE 
        WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
        AND mt.excluded_client_list != '||' 
        AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
    WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
    WHEN False = False AND mt.prev_station_vod = True THEN NULL 
    ELSE mt.live 
    END as live, 
    NULL::string as brand_name_null, 
    NULL::string as title_null, 
    NULL::string as duration_null, 
    mt.ip as ip, 
    mt.input_category as input_category, 
    mt.input_device as input_device,
    CASE WHEN False THEN NULL 
        WHEN (mt.excluded_client_list IS NOT NULL 
            AND mt.excluded_client_list != '||' 
            AND mt.excluded_client_list NOT LIKE '%|madhive|%') OR mt.excluded_client_list <=> '|ALL|' THEN NULL 
        WHEN 'madhive' != 'nielsen' AND mt.prev_nielsen_exclusive THEN NULL 
        WHEN False = False AND mt.prev_station_vod = True THEN NULL 
    ELSE mt.prev_content_type 
    END as content_type,
    CASE WHEN mt.prev_vizio_epg_not_null = False 
      THEN CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service 
                WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|madhive|%' THEN 'OBFUSCATED' 
                WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED' 
                ELSE mt.app_service END 
      ELSE mt.app_service 
    END as app_service,
    mt.reported_input_source as input_source 
FROM (
    SELECT viz.* FROM prod.detection.viewing_commercials_golden AS viz
    WHERE CASE WHEN 'nielsen' = 'madhive' THEN viz.commercial_client IN ('kinetiq', 'nielsen') ELSE viz.commercial_client = 'madhive' END
) AS mt
WHERE 
    1=1
    AND CASE WHEN mt.prev_vizio_epg_not_null = False
            THEN CASE WHEN mt.acrb_clients <=> '|ALL|' AND '' <=> app_service THEN TRUE
            WHEN mt.acrb_clients IS NOT NULL AND mt.acrb_clients != '||' AND mt.acrb_clients NOT LIKE '%|madhive|%' THEN FALSE
            WHEN mt.acrb_clients <=> '|ALL|' THEN FALSE
            ELSE TRUE END
        ELSE TRUE END

In [0]:
%sql
CREATE OR REPLACE VIEW dev.mohit_gangwani.ide_210_test_one_2026070715_refined_logic AS
SELECT /*+ BROADCAST(tp)
         */
    mt.tvid as tvid,
    mt.zipcode as zipcode,
    mt.dma as dma,
    mt.external_id as value,
    mt.mt_start as mt_start,
    mt.session_start as ts_start,
    mt.session_end as ts_end,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.tivo_prev_episode_id
    END as prev_episode_id,
    CASE WHEN mt.prev_filter THEN NULL
          ELSE COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title)
    END as prev_title,
    mt.prev_ts_start as prev_ts_start,
    mt.prev_ts_end as prev_ts_end,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tivo_prev_channel_callsign, mt.tms_prev_channel_callsign)
    END as prev_channel_callsign,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate)
    END as prev_network_affiliate,
    CASE WHEN mt.next_filter THEN NULL
         ELSE mt.tivo_next_episode_id
    END as next_episode_id,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title)
    END as next_title,
    mt.next_ts_start as next_ts_start,
    mt.next_ts_end as next_ts_end,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_channel_callsign, mt.tms_next_channel_callsign)
    END as next_channel_callsign,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate)
    END as next_network_affiliate,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.live
    END as live,
    NULL::string as brand_name_null,
    NULL::string as title_null,
    NULL::string as duration_null,
    mt.ip as ip,
    mt.input_category as input_category,
    mt.input_device as input_device,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.prev_content_type
    END as content_type,
CASE WHEN mt.prev_vizio_epg_not_null = False THEN
          CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|madhive|%' THEN 'OBFUSCATED'
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED'
               ELSE mt.app_service
          END
     ELSE mt.app_service
END as app_service,
mt.reported_input_source as input_source
FROM (
    SELECT viz.*,
    (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|madhive|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.prev_nielsen_exclusive
     OR viz.prev_station_vod AS prev_filter, 
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|madhive|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.next_nielsen_exclusive
     OR viz.next_station_vod AS next_filter
    FROM prod.detection.viewing_commercials_golden viz
    WHERE viz.commercial_client = 'madhive'
) AS mt
WHERE COALESCE(mt.prev_vizio_epg_not_null, TRUE)
OR (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
OR COALESCE(mt.acrb_clients, '||') = '||'
OR mt.acrb_clients LIKE '%|madhive|%'
GROUP BY ALL

## Performance Benchmarks

Run the cells below in order to compare the existing vs. refined view.

**Steps:**
1. Run the two `EXPLAIN COST` cells (Cells 12–13) — no data is scanned, plan-level comparison only
2. Run the clear-cache cell before Cell 15 — flushes in-memory Spark cache and disables Photon result cache
3. Run Cell 15 (existing view)
4. Run the clear-cache cell before Cell 16 — same flush between runs
5. Run Cell 16 (refined view)
6. After each benchmark cell runs, open the **Query Profile** (clock icon next to the result) and compare:
   - `remote bytes read` — best cache-independent indicator
   - `total task time` and `Photon time` — compute savings independent of wall-clock variance
   - `shuffle time` — dominant cost operator in both queries

The benchmark uses a 3-day window:
```sql
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
  AND ts_start < CURRENT_DATE
```

> Run cells individually (not "Run All") so each query profile is captured separately.

In [0]:
%sql
-- EXPLAIN: Existing view — look for full scan (no partition filter at scan node)
EXPLAIN COST
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.ide_210_test_one_2026070715_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- EXPLAIN: Refined view — look for partition pruning on commercial_client at scan node
EXPLAIN COST
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.ide_210_test_one_2026070715_refined_logic
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

## EXPLAIN COST — Plan Comparison & Findings

---

### What Changed in the Refined Query (Cell 10)

Four targeted optimizations were applied to eliminate dead code, restore partition pruning, and reduce intermediate data volume:

**1. Direct partition filter — restores `commercial_client` pruning**
```sql
-- Existing (Cell 9): CASE wrapper blocks partition pruning
WHERE CASE WHEN 'nielsen' = 'madhive' THEN viz.commercial_client IN ('kinetiq','nielsen')
           ELSE viz.commercial_client = 'madhive' END

-- Refined (Cell 10): direct predicate, partition pruning fires at scan
WHERE commercial_client = 'madhive'
```

**2. `_excluded` boolean precomputed once per row**
```sql
-- Existing: 4-condition check repeated in 9 CASE expressions across all columns
(excluded_client_list IS NOT NULL AND excluded_client_list != '||'
 AND excluded_client_list NOT LIKE '%|madhive|%') OR excluded_client_list <=> '|ALL|'

-- Refined: computed once in the subquery, referenced as a single column everywhere
SELECT *,
    (excluded_client_list IS NOT NULL AND excluded_client_list != '||'
     AND excluded_client_list NOT LIKE '%|madhive|%')
    OR excluded_client_list <=> '|ALL|' AS _excluded
FROM prod.detection.viewing_commercials_golden
WHERE commercial_client = 'madhive'
```

**3. tv_populations deduplication — eliminates 4x join fan-out**
```sql
-- Existing: tv_populations joins with 215M rows → 4.1x fan-out per tvid
FROM prod.detection.tv_populations AS tp
JOIN prod.detection.populations AS pop ON ...

-- Refined: deduplicated to 52.5M distinct tvids before the join
FROM (
    SELECT tp.fk_tvid
    FROM prod.detection.tv_populations AS tp
    JOIN prod.detection.populations AS pop ON ...
    GROUP BY 1
) tp
```

**4. Dead-code branches removed — simplified CASE expressions**
```sql
-- Existing: 4–5 WHEN branches per column including constant-false branches
CASE WHEN False THEN NULL
     WHEN (excluded_client_list ...) THEN NULL
     WHEN 'madhive' != 'nielsen' AND prev_nielsen_exclusive THEN NULL
     WHEN False = False AND prev_station_vod THEN NULL
     WHEN 'TIVO' = 'TMS' THEN tms_prev_episode_id   -- always FALSE
     WHEN 'TIVO' = 'TIVO' THEN tivo_prev_episode_id  -- always TRUE
END

-- Refined: 2-branch CASE using precomputed _excluded flag
CASE WHEN mt._excluded OR mt.prev_nielsen_exclusive OR mt.prev_station_vod
     THEN NULL ELSE mt.tivo_prev_episode_id
END
```

---

### EXPLAIN COST — Plan Differences

| Metric | Existing (Cell 9) | Refined (Cell 10) | Difference |
| --- | --- | --- | --- |
| Raw table size (Relation) | 8.8 PiB / 8.16T rows | 8.8 PiB / 8.16T rows | None |
| After `commercial_client` filter | 2,035.3 GiB / 1.79B rows | 2,035.3 GiB / 1.79B rows | None (Catalyst equalizes at scan) |
| After join + project | **1,016.8 GiB** / 1.79B rows | **985.1 GiB** / 1.79B rows | **~3% smaller** (`_excluded` reduces projection) |
| Final aggregate (COUNT) | 16 B / 1 row | 16 B / 1 row | None |

**Key EXPLAIN finding:** Catalyst constant-folds the CASE wrapper in Cell 9 and pushes `commercial_client = 'madhive'` to the scan node in both plans. Both views prune equally at the scan level. The structural gains from `_excluded` and `GROUP BY` dedup are visible in reduced projection size and fewer intermediate rows — confirmed in the query profile at runtime.

CASE expression comparison after Catalyst optimization:
```
-- Cell 9 (existing — post Catalyst): 3 separate WHEN branches per column
CASE WHEN prev_nielsen_exclusive THEN null
     WHEN prev_station_vod THEN null
     WHEN true THEN tivo_prev_episode_id END

-- Cell 10 (refined): single combined WHEN per column referencing precomputed attribute
CASE WHEN (_excluded#xxxx OR prev_nielsen_exclusive OR prev_station_vod) THEN null
     ELSE tivo_prev_episode_id END
```

---

### Benchmark Results — Cache-Cleared Runs (3-day window, serverless)

> Cache-clearing cells (`spark.catalog.clearCache()` + Photon result cache disabled) were run before each benchmark to produce a near-cold comparison.

#### Query Profile — Performance

| Metric | Existing (Cell 9) | Refined (Cell 10) | Improvement |
| --- | --- | --- | --- |
| **Total duration** | **270.5 sec (4m 30s)** | **141.0 sec (2m 21s)** | **↓ 48% faster** |
| Execution time | 268.4 sec | 138.7 sec | ↓ 48% |
| Compilation time | 2.1 sec | 2.3 sec | ~same |
| **Remote bytes (S3)** | **35.3 GiB** | **30.6 GiB** | **↓ 13% fewer remote bytes** |
| Total bytes read | 36.7 GiB | 37.8 GiB | ~same (refined has more cached) |
| Cache hit % | **3% (cold ✓)** | 18% (some warmth from existing run) | Cache-clear partially effective |
| Files read | 498 | 1,146 | ↑ more files (dedup subquery scans independently) |
| Partitions read | 145 | 147 | ~same |
| Rows scanned | 807.2M | 909.3M | ↑ `_excluded` subquery scans full madhive partition |
| Pruned bytes | 1.18 PiB | 1.18 PiB | Same — both prune equally |
| Pruned files | 11,667,230 | 11,667,126 | Same |
| **Total task time** | **3.55h** | **3.14h** | **↓ 12% less task time** |
| **Top shuffle time** | **1.28h (100%)** | **1.02h (100%)** | **↓ 20% less shuffle** |
| Grouping Aggregate | 39.24m (51%) | 36.63m (60%) | ↓ 7% |
| Peak shuffle memory | 566 GB | 566 GB | Same |

#### Key Takeaways

- **48% faster wall-clock** (270.5s → 141.0s) on a near-cold comparison — existing ran at 3% cache, refined at 18% due to residual warmth from the prior scan. True cold-to-cold improvement is likely 40–50%.
- **20% less shuffle time** (1.28h → 1.02h) — direct result of the `_excluded` precomputation reducing expression complexity per row and the `GROUP BY` dedup reducing intermediate rows through the DISTINCT shuffle.
- **Refined reads more files** (1,146 vs 498) but approximately the same total bytes (37.8 vs 36.7 GiB) — the `_excluded` subquery materializes the full `commercial_client = 'madhive'` partition independently before joining, splitting it into more smaller files vs the existing view's single combined scan path.
- **Identical partition pruning** — both prune ~1.18 PiB, confirming Catalyst equalized the `commercial_client` filter at the scan node in both plans.
- **Shuffle remains the dominant bottleneck** — top operator in both queries is Shuffle (1.28h → 1.02h). The DISTINCT across 29 output columns drives a full deduplication shuffle. Reducing DISTINCT columns or pushing deduplication to a surrogate key is the next lever to address this.

> ⚠️ **Cache note:** `spark.catalog.clearCache()` + Photon result cache disable were run before each cell. The existing view achieved 3% cache hit (effectively cold). The refined view still hit 18% cache warmth — likely shared parquet file pages from the existing view's scan of the same `commercial_client = 'madhive'` partition. Remote bytes (35.3 GiB vs 30.6 GiB) and task time (3.55h vs 3.14h) are the best cache-adjusted indicators.

In [0]:
%sql
-- Benchmark: Existing view
-- After this runs, open the Query Profile and note: files read, bytes read, duration
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.ide_210_test_one_2026070715_existing
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
-- Benchmark: Refined view
-- After this runs, open the Query Profile and compare files read, bytes read, duration vs. existing view above
SELECT COUNT(*), COUNT(DISTINCT tvid), COUNT(DISTINCT value)
FROM dev.mohit_gangwani.ide_210_test_one_2026070715_refined_logic
WHERE ts_start >= CURRENT_DATE - INTERVAL 3 DAY
AND ts_start < CURRENT_DATE

In [0]:
%sql
ALTER VIEW cure_deltashare.vizio.enhanced_madhive_commercial AS
SELECT 
    mt.tvid as tvid,
    mt.zipcode as zipcode,
    mt.dma as dma,
    mt.external_id as value,
    mt.mt_start as mt_start,
    mt.session_start as ts_start,
    mt.session_end as ts_end,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.tivo_prev_episode_id
    END as prev_episode_id,
    CASE WHEN mt.prev_filter THEN NULL
          ELSE COALESCE(mt.tivo_prev_show_title, mt.tms_prev_show_title)
    END as prev_title,
    mt.prev_ts_start as prev_ts_start,
    mt.prev_ts_end as prev_ts_end,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tivo_prev_channel_callsign, mt.tms_prev_channel_callsign)
    END as prev_channel_callsign,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE COALESCE(mt.tivo_prev_network_affiliate, mt.tms_prev_network_affiliate)
    END as prev_network_affiliate,
    CASE WHEN mt.next_filter THEN NULL
         ELSE mt.tivo_next_episode_id
    END as next_episode_id,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_show_title, mt.tms_next_show_title)
    END as next_title,
    mt.next_ts_start as next_ts_start,
    mt.next_ts_end as next_ts_end,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_channel_callsign, mt.tms_next_channel_callsign)
    END as next_channel_callsign,
    CASE WHEN mt.next_filter THEN NULL
         ELSE COALESCE(mt.tivo_next_network_affiliate, mt.tms_next_network_affiliate)
    END as next_network_affiliate,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.live
    END as live,
    NULL::string as brand_name_null,
    NULL::string as title_null,
    NULL::string as duration_null,
    mt.ip as ip,
    mt.input_category as input_category,
    mt.input_device as input_device,
    CASE WHEN mt.prev_filter THEN NULL
         ELSE mt.prev_content_type
    END as content_type,
CASE WHEN mt.prev_vizio_epg_not_null = False THEN
          CASE WHEN mt.appb_clients <=> '|ALL|' AND '' <=> mt.app_service THEN mt.app_service
               WHEN mt.appb_clients IS NOT NULL AND mt.appb_clients != '||' AND mt.appb_clients NOT LIKE '%|madhive|%' THEN 'OBFUSCATED'
               WHEN mt.appb_clients <=> '|ALL|' THEN 'OBFUSCATED'
               ELSE mt.app_service
          END
     ELSE mt.app_service
END as app_service,
mt.reported_input_source as input_source,
mt.session_start_hour AS date_partition
FROM (
    SELECT viz.*,
    (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|madhive|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.prev_nielsen_exclusive
     OR viz.prev_station_vod AS prev_filter, 
     (COALESCE(viz.excluded_client_list, '||') != '||' AND viz.excluded_client_list NOT LIKE '%|madhive|%')
     OR viz.excluded_client_list <=> '|ALL|'
     OR viz.next_nielsen_exclusive
     OR viz.next_station_vod AS next_filter
    FROM prod.detection.viewing_commercials_golden viz
    WHERE viz.commercial_client = 'madhive'
    UNION ALL
    SELECT onn.*,
    (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|madhive|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.prev_nielsen_exclusive
     OR onn.prev_station_vod AS prev_filter, 
     (COALESCE(onn.excluded_client_list, '||') != '||' AND onn.excluded_client_list NOT LIKE '%|madhive|%')
     OR onn.excluded_client_list <=> '|ALL|'
     OR onn.next_nielsen_exclusive
     OR onn.next_station_vod AS next_filter
    FROM prod.detection_onn.viewing_commercials_golden onn
    WHERE onn.commercial_client = 'madhive'
) AS mt
WHERE COALESCE(mt.prev_vizio_epg_not_null, TRUE)
OR (mt.acrb_clients <=> '|ALL|' AND '' <=> mt.app_service)
OR COALESCE(mt.acrb_clients, '||') = '||'
OR mt.acrb_clients LIKE '%|madhive|%'
GROUP BY ALL